# Du modèle évalué au POC de triage

Je dispose d'un SFT triage v3 sélectionné sur validation. Le pilote DPO apprend les préférences, mais n'améliore pas le classement : je garde cette variante pour la comparaison expérimentale.

Je cherche maintenant à rendre le modèle utilisable dans une démonstration : **collecter → proposer → vérifier → tracer**. Ce notebook vérifie la partie applicative sur CPU. Les cellules ne chargent aucun poids et ne déclenchent aucun calcul GPU.

In [1]:
from pathlib import Path
import json
import sys

RACINE = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'serving/modele.json').exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
modele = json.loads((RACINE / 'serving/modele.json').read_text())
print({k: modele[k] for k in ('id', 'revision', 'adapter_sha256', 'mlflow_run_id')})

{'id': 'ocr14-sft-triage-v3', 'revision': 'ea980cb0a6c2ae4b936e82123acc929f1cec04c1', 'adapter_sha256': '7ca4d0257eb7cab0bccf9427efabb47f3a66bde3ec8bb9d76567f3cda086b7e5', 'mlflow_run_id': 'a63b597a3afd461588cddeb72375c856'}


## 1. Traduire les attentes du projet en fonctions vérifiables

| Attente | Réalisation | Ce qui reste à valider |
|---|---|---|
| Questionnaire adaptatif FR/EN | Champs manquants et question conditionnelle sur la douleur | Pertinence du recueil ; pas de protocole clinique automatisé |
| Trois priorités et explication | Lecture du texte SFT, vérification ESI/priorité, revue humaine si ambiguïté | Fidélité et qualité des explications produites en situation |
| Traçabilité | Identifiants, horodatage, entrée, réponse, provenance et erreurs dans SQLite | Conservation durable et droits par utilisateur |
| Intégration SIH | Contrat JSON documenté par FastAPI | Aucun raccordement réel ni FHIR |
| Docker / vLLM | Image API et lancement vLLM préparés | Inférence vLLM et latence réelle |
| CI/CD et cloud | Tests + construction Docker dans le workflow CI | Exécution sur GitHub et cible de déploiement |

Je sépare une validation technique de la qualité clinique : un statut HTTP 200 et un ESI lisible ne prouvent pas la justesse de la proposition.

## 2. Recueillir sans inventer d'informations

Le LoRA a appris à classer une présentation, pas à mener un entretien. Je choisis donc des questions déterministes et faciles à auditer. La branche « douleur » est un repérage lexical simple, sans compréhension des négations ; elle ajoute une question, jamais une règle de priorité.

Les réponses inconnues restent explicites. Le remplissage du formulaire ne garantit pas qu'un dossier soit médicalement suffisant. L'ordre et les libellés sont centralisés dans `serving/domaine.py`.

In [2]:
from serving.domaine import Consultation, prochaine_question, presentation

cas = Consultation(langue='fr', plainte='Cas fictif : douleur au poignet.')
reponses_fictives = {'debut': 'Ce matin', 'evolution': 'Stable',
                    'retentissement': 'Gêne pour écrire', 'contexte': 'Personnage de 40 ans ; antécédents inconnus'}
while question := prochaine_question(cas):
    print(question['champ'], '→', question['texte'])
    cas.reponses[question['champ']] = reponses_fictives[question['champ']]
print('\nPrésentation transmise au modèle :\n' + presentation(cas))

debut → Depuis quand les symptômes sont-ils présents ?
evolution → Comment les symptômes évoluent-ils ?
retentissement → Quel est le retentissement de la douleur sur les activités ?
contexte → Quels âge, antécédents ou traitements sont connus ? Indiquer les informations inconnues.

Présentation transmise au modèle :
Cas fictif : douleur au poignet.
Début : Ce matin
Évolution : Stable
Contexte : Personnage de 40 ans ; antécédents inconnus
Retentissement : Gêne pour écrire


## 3. Garder le format appris

```text
Présentation du cas + consigne FR/EN issue du SFT
                    ↓
Tokenizer du socle à la révision figée ; réflexion désactivée
                    ↓
IDs de tokens → /v1/completions de vLLM → EOS natif
                    ↓
Contrôle du format → proposition à valider ou revue humaine
```

Je limite la génération à 192 tokens, sans échantillonnage. Je refuse une entrée dépassant 2 048 tokens : une troncature silencieuse pourrait supprimer un élément du cas. `lancer_vllm.py` vérifie l'empreinte des poids avant de démarrer le serveur.

Les mêmes tokens ne garantissent pas exactement le même résultat avec deux moteurs de calcul. Je vérifierai la transition Transformers → vLLM sur validation avant de figer le test final. Le questionnaire modifie également la présentation des cas : cet effet doit être évalué séparément.

In [3]:
import inspect
from serving.backend import Vllm
print(inspect.getsource(Vllm.generer))

    def generer(self, cas, langue):
        """Refuse les entrées longues puis envoie les IDs exacts, avec décodage glouton."""
        with self.verrou:
            if self.tokenizer is None:
                from transformers import AutoTokenizer
                self.tokenizer = AutoTokenizer.from_pretrained(MODELE['socle'], revision=MODELE['revision'])
        prompt = construire_prompt(self.tokenizer, [
            {'role': 'system', 'content': MODELE['systemes'][langue]}, {'role': 'user', 'content': cas}])
        ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        if len(ids) > MODELE['max_input_tokens']:
            raise EntreeTropLongue('Entrée trop longue ; reformuler sans supprimer un élément utile.')
        assert self.tokenizer.eos_token_id == MODELE['eos_token_id']
        reponse = self.client.post('completions', json={
            'model': MODELE['id'], 'prompt': ids, 'temperature': 0, 'max_tokens': MODELE['max_new_tokens'],
            'stop_token_ids

## 4. Refuser une priorité ambiguë

Je conserve la réponse brute pour l'audit, mais je renvoie `esi=null` et `priorite=null` si elle est tronquée, incohérente ou non analysable. Même une réponse valide reste une proposition à vérifier par un soignant.

In [4]:
from serving.domaine import interpreter

sortie = 'Niveau ESI : 3\nPriorité : urgence maximale\nOrientation : Évaluation médicale.'
resultat = interpreter(sortie, 'stop')
assert resultat['statut'] == 'revue_humaine' and resultat['priorite'] is None
print(json.dumps(resultat, ensure_ascii=False, indent=2))

{
  "statut": "revue_humaine",
  "esi": null,
  "priorite": null,
  "motif": "Sortie incomplète, ambiguë ou incohérente.",
  "texte_modele": "Niveau ESI : 3\nPriorité : urgence maximale\nOrientation : Évaluation médicale.",
  "validation_humaine_requise": true
}


## 5. Vérifier le contrat et la trace sans GPU

Le test suivant utilise un **double de test**, pas le modèle médical. Il vérifie que la réponse rendue et celle conservée dans le journal sont identiques. Le dossier temporaire est détruit à la fin de la cellule.

In [5]:
import tempfile
from fastapi.testclient import TestClient
from serving.api import creer_app
from tests.test_serving import FauxBackend, CLE, CAS

with tempfile.TemporaryDirectory() as dossier:
    app = creer_app(backend=FauxBackend(), cle=CLE, audit_path=Path(dossier) / 'audit.sqlite3')
    with TestClient(app) as client:
        entetes = {'Authorization': f'Bearer {CLE}'}
        retour = client.post('/v1/consultations', json=CAS, headers=entetes)
        assert retour.status_code == 200
        resultat = retour.json()
        trace = client.get('/v1/interactions/' + resultat['interaction_id'], headers=entetes).json()
        assert trace['resultat'] == resultat
        print({'origine': 'double de test, sans inférence', 'statut': resultat['statut'],
               'trace_identique': True, 'validation_humaine': resultat['validation_humaine_requise']})

{'origine': 'double de test, sans inférence', 'statut': 'proposition_a_valider', 'trace_identique': True, 'validation_humaine': True}


Le journal conserve aussi les erreurs et les interactions restées `en_cours` après une interruption. Si l'écriture échoue, l'API ne délivre pas de proposition.

**Limites du POC :** SQLite contient les textes fictifs en clair, la clé d'accès est partagée, le journal n'est pas immuable. Je n'en déduis pas une conformité RGPD ni une autorisation d'utiliser des données de patients.

In [6]:
import unittest
suite = unittest.defaultTestLoader.loadTestsFromNames(['tests.test_serving', 'tests.test_space'])
verification = unittest.TextTestRunner(verbosity=1).run(suite)
assert verification.wasSuccessful()


.................
----------------------------------------------------------------------
Ran 17 tests in 0.443s

OK


## 6. Conteneuriser et vérifier le parcours

L'image API a été construite localement. Un premier essai en conteneur a révélé l'absence de Jinja2, nécessaire au template du tokenizer ; la dépendance et un test dédié ont été ajoutés.

Le parcours navigateur a vérifié la collecte et la consultation de l'audit avec un backend indisponible. Ces contrôles ne mesurent pas la vitesse du modèle. `serving/verifier_endpoint.py` prépare trois requêtes fictives et mesure leur latence HTTP lorsque l'inférence réelle est disponible.

In [7]:
# Commandes reproductibles ; aucune allocation GPU automatique dans ce notebook.
# python -m pip install -r serving/requirements.txt
# docker build -f serving/Dockerfile -t ocr14-api:local .
# docker compose -f serving/compose.yaml --profile gpu up --build -d
# python -m serving.verifier_endpoint --output data/evaluation/poc/verification_endpoint.json
# docker compose -f serving/compose.yaml --profile gpu down

print('Paramètres à fournir : OCR14_API_KEY, VLLM_API_KEY, OCR14_ADAPTER_DIR.')
print('Les clés restent dans un environnement privé ; les poids sélectionnés ont été récupérés depuis Drive et vérifiés par SHA256.')

Paramètres à fournir : OCR14_API_KEY, VLLM_API_KEY, OCR14_ADAPTER_DIR.
Les clés restent dans un environnement privé ; les poids sélectionnés ont été récupérés depuis Drive et vérifiés par SHA256.


## 7. Héberger la démonstration sans GPU permanent

Je choisis **Gradio + Transformers/PEFT sur ZeroGPU** pour une démonstration gratuite. Je conserve FastAPI/Docker/vLLM pour vérifier séparément l'exigence de déploiement du cahier des charges. Je n'assimile pas ces deux preuves.

```text
Formulaire FR/EN sur CPU → champs manquants → aucun appel GPU
                         → cas complet → GPU alloué → génération → GPU libéré
Chaque étape → trace de la session → export JSON à télécharger
```

Je conserve les poids, le prompt, l'EOS et le décodage glouton du SFT retenu. Le runtime ZeroGPU utilise PyTorch 2.13.0, compatible avec cette infrastructure ; une identité numérique des sorties avec Colab n'est pas démontrée.

| Élément | Choix | Limite |
|---|---|---|
| Modèle | SFT triage v3, SHA256 vérifié après récupération depuis Drive | Pas de nouvel entraînement |
| Sessions | État isolé par visiteur | Pas d'identité soignant ni d'archivage patient |
| Traces | Export JSON, cache temporaire | Télécharger avant fermeture ; pas d'immutabilité |
| Ressources | ZeroGPU uniquement | Quotas quotidiens et attente possibles |
| Publication | Liste fermée de code et poids | Aucun corpus, secret ou résultat individuel envoyé |

La création de [mon Space](https://huggingface.co/spaces/rayakevin/ocr14-triage-fr-en) a été acceptée avec le matériel `zero-a10g` (nom de l'offre dans l'API). Le paquet est publié et le Space fonctionne : deux cas fictifs FR/EN ont été exécutés de bout en bout.
[Conditions ZeroGPU vérifiées le 5 novembre 2026](https://huggingface.co/docs/hub/spaces-zerogpu).

### Constats lors du déploiement

| Observation | Correction | Enseignement |
|---|---|---|
| Les extensions Gradio installées par Spaces imposent Pydantic ≤ 2.12.5 | Dépendances propres au Space ; tests CPU rejoués avec 2.12.5 | Un environnement local fonctionnel ne garantit pas une construction hébergée |
| PEFT déduit CUDA au chargement alors que le GPU ZeroGPU n'est pas encore alloué | Lecture du LoRA avec `torch_device="cpu"`, puis préparation CUDA au niveau du module | Chargement des poids et calcul GPU sont deux étapes distinctes |
| La trace de session doit survivre à la fermeture de la page | Export JSON téléchargeable, cache temporaire | Afficher une trace ne constitue pas un archivage durable |


### 7.1 Vérifier la collecte et la trace avant l'inférence

Je réutilise `serving/domaine.py` dans `serving/space/parcours.py`. Ce contrôle utilise un **double de test** : il vérifie un seul appel final et l'export, sans produire de résultat médical réel.


Je propose aussi un mode « Cas complet » et des détails techniques facultatifs. Le sélecteur en en-tête traduit les textes ; changer de langue ou de mode réinitialise le cas. Les paramètres de génération restent fixes.


In [8]:
from serving.space.parcours import nouvelle_session, avancer, exporter_trace
from tests.test_serving import FauxBackend

moteur = FauxBackend()
etat = nouvelle_session('fr')
for reponse in ('Cas fictif : douleur du poignet.', 'Depuis hier', 'Stable', 'Activités limitées'):
    etat = avancer(reponse, etat, moteur.generer)
assert moteur.appels == 0
etat = avancer('Contexte inconnu', etat, moteur.generer)
assert moteur.appels == 1
trace = json.loads(exporter_trace(etat))
print({'etapes': len(trace['evenements']), 'appels_simules': moteur.appels,
       'statut_simule': trace['evenements'][-1]['reponse']['statut']})


{'etapes': 5, 'appels_simules': 1, 'statut_simule': 'proposition_a_valider'}


### 7.2 Préparer un paquet reproductible

`serving/space/interface.py` construit Gradio ; `app.py` charge le modèle et décore seulement la génération avec `spaces.GPU`. `publier.py` vérifie l'empreinte du LoRA et assemble une liste explicite de fichiers. Je prépare le paquet sur CPU ; l'appel externe de publication reste commenté.


In [9]:
from serving.space.publier import preparer

adaptateur = RACINE / 'models/sft_triage_v3/adapter'
if adaptateur.exists():
    with tempfile.TemporaryDirectory() as dossier:
        empreintes = preparer(adaptateur, Path(dossier) / 'space')
    assert empreintes['adapter/adapter_model.safetensors'] == modele['adapter_sha256']
    print(f'{len(empreintes)} fichiers autorisés ; empreinte du LoRA conforme.')
else:
    print('Récupérer le LoRA SFT v3 depuis Drive avant de reconstruire le paquet.')

# Publication externe : uniquement après vérification du paquet et de l'offre gratuite.
# python -m serving.space.publier --adapter models/sft_triage_v3/adapter \
#   --destination data/deployment/nouveau_paquet --publier COMPTE/NOUVEAU_SPACE
# Un Space existant n'est jamais écrasé par cette commande de création.


12 fichiers autorisés ; empreinte du LoRA conforme.


### 7.3 Vérifier une interaction réelle en ligne

J'ai exécuté deux parcours depuis le navigateur : français avec la branche douleur, anglais sans cette branche. Les réponses du vrai modèle sont correctement structurées et les deux traces JSON ont été récupérées. Les durées ci-dessous sont mesurées côté serveur pendant l'étape finale, allocation ZeroGPU comprise ; deux essais ne constituent pas un benchmark.

Je distingue **fonctionnement technique** et **pertinence clinique** : je n'utilise pas ces cas fictifs comme vérité terrain et je ne remplace pas le test final du corpus par ces essais.


Après la mise à jour de l’interface, je vérifie les traductions et les réglages. Une troisième tentative, en saisie directe, atteint le quota invité ZeroGPU : aucune priorité n’est délivrée. Ce résultat vérifie le traitement d’une indisponibilité, pas une nouvelle génération réussie.


In [10]:
verification = RACINE / 'data/evaluation/poc/verification_space_reelle.json'
if verification.exists():
    bilan = json.loads(verification.read_text())
    print(bilan['url'])
    print('Commit vérifié :', bilan['commit'])
    for essai in bilan['essais']:
        print(f"{essai['langue']} : {essai['etapes']} étapes ; "
              f"{essai['tokens_sortie']} tokens ; {essai['duree_s']:.3f} s ; {essai['statut']}")
else:
    print('Traces des essais en ligne absentes de cette copie ; aucune inférence relancée.')


https://huggingface.co/spaces/rayakevin/ocr14-triage-fr-en
Commit vérifié : 5f4f2ab92684e271c407ea6e7438e752708d2a43
fr : 5 étapes ; 68 tokens ; 5.115 s ; proposition_a_valider
en : 4 étapes ; 53 tokens ; 3.870 s ; proposition_a_valider


## 8. Ce que je peux conclure à ce stade

- Les 17 tests CPU passent : contrat API, collecte, isolation, erreurs et traces.
- Le LoRA récupéré correspond au SFT v3 retenu ; le paquet publié ne contient aucun corpus.
- La génération ZeroGPU est vérifiée sur deux cas fictifs FR/EN, avec export des traces.
- Les essais techniques n'établissent pas la pertinence clinique. Le test final reste réservé.
- La vérification réelle vLLM, les mesures de latence et l'effet du questionnaire demeurent des preuves distinctes à produire.


### Clôture du projet : critères encore ouverts

| Attendu | État constaté | Action de clôture |
|---|---|---|
| Corpus SFT/DPO bilingue | Partitions v3 et provenance documentées | Finaliser la présentation des licences et limites de validation |
| SFT LoRA puis DPO | SFT sélectionné ; pilote DPO comparé | Justifier le maintien du SFT et le périmètre du pilote |
| Évaluation indépendante | Choix faits sur validation | Évaluer les 226 entrées triage de test restantes, sans réajuster le modèle sur ce test |
| Endpoint vLLM cloud | API/Docker préparés ; démo ZeroGPU fonctionnelle avec Transformers | Exécuter et mesurer la voie vLLM ou faire accepter explicitement l’écart |
| CI/CD GitHub Actions | Workflow de tests et construction présent localement | Versionner, exécuter le pipeline et ajouter le déploiement automatisé |
| Rapport technique ≤ 20 pages | Démarche répartie dans les notebooks | Rédiger la synthèse, les limites et la trajectoire de passage à l’échelle |

Je ne considère donc pas le projet comme terminé. Je peux avancer vers la
validation finale et les livrables sans relancer systématiquement les entraînements.
Les annotations restent un standard silver et les erreurs de sous-triage observées
interdisent de revendiquer une validation clinique. L’intégration SIH reste simulée.


## CI/CD effectivement exécutée

Le [run GitHub Actions 42074793162](https://github.com/rayakevin/OCR_PROJET_14/actions/runs/42074793162) réussit les **35 tests CI**, la **construction Docker** et la **publication sur le Space gratuit existant**. La révision retournée est vérifiée en fonctionnement ; un paquet identique ne crée pas nécessairement un nouveau commit.

Le déploiement utilise le secret `HF_TOKEN` d’Actions et une liste fermée de fichiers. Les corpus, journaux et secrets restent exclus. Cette preuve concerne le Space Transformers ; elle ne vaut pas vérification du moteur vLLM.

La vérification vLLM préparée dans le [notebook 05](../evaluation/05_finalisation_colab.ipynb) reste à exécuter sur GPU. Son endpoint est temporaire et interne à Colab. Je conserve cette limite dans le rapport.


## Bilan de la vérification réelle vLLM/API

Je vérifie le SFT retenu sur une A100 40 Go, avec vLLM 0.32.0. Le premier essai échoue car le port 8080 est déjà occupé. Je réserve ensuite un port libre transmis à Uvicorn ; un test CPU reproduit le conflit, puis le notebook 06 vérifie réellement l'inférence.

| Requête fictive | Statut | Temps HTTP |
|---|---|---:|
| FR, gorge | Proposition à valider | 6,935 s |
| EN, gorge | Revue humaine | 0,385 s |
| FR, poignet | Proposition à valider | 0,514 s |

La médiane est 0,514 s. Le premier appel peut inclure des initialisations ; trois requêtes séquentielles ne permettent pas de conclure sur la charge. Les trois traces SQLite sont concordantes. Les contrôles produisent 401 sans clé, 422 pour une entrée invalide et une question quand le cas est incomplet.

**Pourquoi la revue humaine ?** Le modèle écrit « delayed emergency » ; le contrat attend « delayed urgency ». Le niveau ESI seul ne suffit pas : l'API laisse la priorité à null. Je conserve cette limite observée sans modifier l'interprétation après l'essai. Une future normalisation doit être testée séparément sur des cas ambigus et contradictoires.

Les preuves sont dans `data/colab_archive/vllm_reprise_20261105_211011/`. La VM est temporaire et les appels internes : le Space public reste une démonstration Transformers, pas un endpoint vLLM permanent. Aucun résultat ne constitue une validation clinique.
